In [42]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats  
import statsmodels.api as sm
import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings('ignore')

In [43]:
# load data
data = pd.read_csv('data/campaign_data.csv')

data.head()

,Client ID,Client Type,Number of Customers,Montly Target,Zip Code,Calendardate,Amount Collected,Unit Sold,Campaign (Email),Campaign (Flyer),Campaign (Phone),Sales Contact 1,Sales Contact 2,Sales Contact 3,Sales Contact 4,Sales Contact 5,Number of Competition
0,ID-987275,Medium Facility,2800,125,1003,16-01-2014,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
1,ID-987275,Medium Facility,2800,125,1003,16-02-2014,3409460,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,322500.0,Low
2,ID-987275,Medium Facility,2800,125,1003,18-03-2014,10228384,75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
3,ID-987275,Medium Facility,2800,125,1003,18-04-2014,17047304,123,0.0,0.0,0.0,0.0,3547500.0,1290000.0,0.0,0.0,Low
4,ID-987275,Medium Facility,2800,125,1003,19-05-2014,23866224,171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low


In [44]:
data.columns

Index(['Client ID', 'Client Type', 'Number of Customers', 'Montly Target',
       'Zip Code', 'Calendardate', 'Amount Collected', 'Unit Sold',
       'Campaign (Email)', 'Campaign (Flyer)', 'Campaign (Phone)',
       'Sales Contact 1', 'Sales Contact 2', 'Sales Contact 3',
       'Sales Contact 4', 'Sales Contact 5', 'Number of Competition'],
      dtype='object')

In [45]:
data.dtypes

Client ID                 object
Client Type               object
Number of Customers        int64
Montly Target              int64
Zip Code                   int64
Calendardate              object
Amount Collected           int64
Unit Sold                  int64
Campaign (Email)         float64
Campaign (Flyer)         float64
Campaign (Phone)         float64
Sales Contact 1          float64
Sales Contact 2          float64
Sales Contact 3          float64
Sales Contact 4          float64
Sales Contact 5          float64
Number of Competition     object
dtype: object

In [46]:
# convert to datetime variables
data['Calendardate'] = pd.to_datetime(data['Calendardate'])
data['Calendar_Month'] = data['Calendardate'].dt.month
data['Calendar_Year'] = data['Calendardate'].dt.year

In [47]:
data['Client Type'].value_counts(normalize=True)

Client Type
Large Facility      0.459677
Small Facility      0.282258
Medium Facility     0.169355
Private Facility    0.088710
Name: proportion, dtype: float64

In [48]:
pd.crosstab(data['Number of Competition'], 
            data['Client Type'], 
            margins = True,
            normalize = 'columns')

Client Type,Large Facility,Medium Facility,Private Facility,Small Facility,All
Number of Competition,,,,,
High,0.166667,0.166667,0.166667,0.166667,0.166667
Low,0.833333,0.833333,0.833333,0.833333,0.833333


In [49]:
data.groupby('Number of Competition').mean(numeric_only=True)

,Number of Customers,Montly Target,Zip Code,Amount Collected,Unit Sold,Campaign (Email),Campaign (Flyer),Campaign (Phone),Sales Contact 1,Sales Contact 2,Sales Contact 3,Sales Contact 4,Sales Contact 5,Calendar_Month,Calendar_Year
Number of Competition,,,,,,,,,,,,,,,
High,1456.935484,75.080645,1003.0,2.974789e+07,213.127016,105398.938508,994046.717540,45198.036895,146945.564516,2.685333e+06,1.786754e+06,72172.379032,8452.620968,10.5,2015.0
Low,1456.935484,75.080645,1003.0,1.445570e+07,103.132258,150862.165766,623692.979839,26693.304194,128219.758065,1.890916e+06,1.883634e+06,70481.854839,15864.919355,5.7,2014.4


In [50]:
data.groupby('Client Type').mean(numeric_only=True)

,Number of Customers,Montly Target,Zip Code,Amount Collected,Unit Sold,Campaign (Email),Campaign (Flyer),Campaign (Phone),Sales Contact 1,Sales Contact 2,Sales Contact 3,Sales Contact 4,Sales Contact 5,Calendar_Month,Calendar_Year
Client Type,,,,,,,,,,,,,,,
Large Facility,1380.842105,71.578947,1003.0,1.999880e+07,143.098684,142273.609649,8.192056e+05,45595.436623,133667.763158,2.034013e+06,2.017039e+06,119287.280702,16266.447368,6.5,2014.5
Medium Facility,3940.761905,202.857143,1003.0,4.075997e+07,290.583333,437217.097817,1.552603e+06,49176.847619,398645.833333,4.822783e+06,4.698646e+06,85104.166667,33273.809524,6.5,2014.5
Private Facility,400.727273,20.454545,1003.0,5.030246e+06,35.784091,5183.715152,2.272919e+05,5522.470455,1221.590909,6.376705e+05,4.434375e+05,3664.772727,12215.909091,6.5,2014.5
Small Facility,422.514286,21.285714,1003.0,1.637759e+06,11.689286,11975.986310,9.120875e+04,0.000000,8062.500000,7.617143e+05,3.727946e+05,4223.214286,1535.714286,6.5,2014.5


In [51]:
data.corr(numeric_only=True)['Amount Collected']

Number of Customers    0.607496
Montly Target          0.608204
Zip Code                    NaN
Amount Collected       1.000000
Unit Sold              0.997515
Campaign (Email)       0.248235
Campaign (Flyer)       0.444337
Campaign (Phone)       0.034858
Sales Contact 1        0.277478
Sales Contact 2        0.552112
Sales Contact 3        0.357887
Sales Contact 4        0.236165
Sales Contact 5        0.095795
Calendar_Month         0.139425
Calendar_Year          0.286194
Name: Amount Collected, dtype: float64

In [52]:
cm = sns.light_palette('green', as_cmap=True)

correlation_analysis = pd.DataFrame(data[['Amount Collected', 'Campaign (Email)',
                                        'Campaign (Flyer)', 'Campaign (Phone)',
                                        'Sales Contact 1', 'Sales Contact 2',
                                        'Sales Contact 3', 'Sales Contact 4',
                                        'Sales Contact 5']]
                                        .corr()['Amount Collected']
                                        .reset_index())
correlation_analysis.columns = ['Impact Variable', 'Degree of Linear Impact (Correlation)']
correlation_analysis = correlation_analysis[correlation_analysis['Impact Variable'] != 'Amount Collected']
correlation_analysis = correlation_analysis.sort_values('Degree of Linear Impact (Correlation)', ascending=False)
correlation_analysis.style.background_gradient(cmap=cm).format({'Degree of Linear Impact (Correlation)': '{:.2f}'})

,Impact Variable,Degree of Linear Impact (Correlation)
5,Sales Contact 2,0.55
2,Campaign (Flyer),0.44
6,Sales Contact 3,0.36
4,Sales Contact 1,0.28
1,Campaign (Email),0.25
7,Sales Contact 4,0.24
8,Sales Contact 5,0.10
3,Campaign (Phone),0.03


In [54]:
sns.light_palette('green', as_cmap=True)

correlation_analysis = pd.DataFrame(data.groupby('Client Type')[['Amount Collected', 'Campaign (Email)',
                                        'Campaign (Flyer)', 'Campaign (Phone)',
                                        'Sales Contact 1', 'Sales Contact 2',
                                        'Sales Contact 3', 'Sales Contact 4',
                                        'Sales Contact 5']]
                                        .corr()['Amount Collected']
                                        .reset_index())
correlation_analysis.columns = ['Acc Type', 'Impact Variable', 'Impact']
correlation_analysis = correlation_analysis[correlation_analysis['Impact Variable'] != 'Amount Collected']
correlation_analysis = correlation_analysis.sort_values(['Acc Type', 'Impact'], ascending=False)
correlation_analysis.style.background_gradient(cmap=cm).format({'Impact': '{:.2f}'})

,Acc Type,Impact Variable,Impact
32,Small Facility,Sales Contact 2,0.22
33,Small Facility,Sales Contact 3,0.07
28,Small Facility,Campaign (Email),0.06
29,Small Facility,Campaign (Flyer),0.04
34,Small Facility,Sales Contact 4,0.02
35,Small Facility,Sales Contact 5,0.00
31,Small Facility,Sales Contact 1,-0.02
30,Small Facility,Campaign (Phone),nan
23,Private Facility,Sales Contact 2,0.57
20,Private Facility,Campaign (Flyer),0.28


In [55]:
data.columns=[str.replace(' ', '_') for str in data.columns]
data.columns=[str.replace('(', '') for str in data.columns]
data.columns=[str.replace(')', '') for str in data.columns]
results = smf.ols('Amount_Collected ~ Campaign_Email + Campaign_Flyer + Campaign_Phone + Sales_Contact_1 + Sales_Contact_2 + Sales_Contact_3 + Sales_Contact_4 + Sales_Contact_5', 
                  data=data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:       Amount_Collected   R-squared:                       0.480
Model:                            OLS   Adj. R-squared:                  0.478
Method:                 Least Squares   F-statistic:                     342.1
Date:                Sun, 01 Mar 2026   Prob (F-statistic):               0.00
Time:                        07:40:15   Log-Likelihood:                -54512.
No. Observations:                2976   AIC:                         1.090e+05
Df Residuals:                    2967   BIC:                         1.091e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        1.481e+06   5.12e+05     

In [56]:
df = pd.read_html(results.summary().tables[1].as_html(), header=0, index_col=0)[0]
df = df.reset_index()
df = df[df['P>|t|'] < 0.05][['index','coef']]
df

,index,coef
0,Intercept,1.481000e+06
2,Campaign_Flyer,3.337600e+00
4,Sales_Contact_1,4.236800e+00
5,Sales_Contact_2,3.638200e+00
6,Sales_Contact_3,2.343200e+00
7,Sales_Contact_4,1.094780e+01


In [60]:
consolidated_summary = pd.DataFrame()

for acctype in list(set(list(data['Client_Type']))):
    temp_data = data[data['Client_Type'] == acctype].copy()
    results = smf.ols('Amount_Collected ~ Campaign_Email + Campaign_Flyer + Campaign_Phone + Sales_Contact_1 + Sales_Contact_2 + Sales_Contact_3 + Sales_Contact_4 + Sales_Contact_5', 
                      data=temp_data).fit()
    df = pd.read_html(results.summary().tables[1].as_html(), header=0, index_col=0)[0].reset_index()
    df = df[df['P>|t|'] < 0.05][['index','coef']]
    df['Account Type'] = acctype
    df = df.sort_values('coef', ascending=False)
    df = df[df['index'] != 'Intercept']
    print(acctype)
    consolidated_summary = pd.concat([consolidated_summary, df], ignore_index=True)
    print(df)

Private Facility
             index    coef      Account Type
5  Sales_Contact_2  6.6223  Private Facility
Large Facility 
             index     coef     Account Type
4  Sales_Contact_1  11.6731  Large Facility 
7  Sales_Contact_4  10.6145  Large Facility 
5  Sales_Contact_2   4.0031  Large Facility 
2   Campaign_Flyer   2.7204  Large Facility 
6  Sales_Contact_3   2.0316  Large Facility 
3   Campaign_Phone  -3.5361  Large Facility 
Small Facility 
             index      coef     Account Type
5  Sales_Contact_2  0.810100  Small Facility 
3   Campaign_Phone -0.000006  Small Facility 
Medium Facility
             index    coef     Account Type
2   Campaign_Flyer  4.1059  Medium Facility
5  Sales_Contact_2  3.5778  Medium Facility
4  Sales_Contact_1  3.1365  Medium Facility
6  Sales_Contact_3  2.1174  Medium Facility
